In [9]:
import json
import re
import unicodedata
from collections import Counter, defaultdict
from html import unescape

In [10]:
with open('laptops_v2.json', 'r', encoding='utf-8') as file:
    laptops = json.load(file)

In [11]:
print(len(laptops))
print(len(laptops[0]))
print(list(laptops[0].keys()))

380
36
['product_id', 'name', 'sku', 'brand', 'price_original', 'price_sale', 'current_price', 'discount_percent', 'price_segment', 'cpu', 'ram', 'ram_gb', 'ram_type', 'storage', 'storage_gb', 'storage_max_gb', 'storage_upgradable', 'gpu', 'gpu_type', 'screen_size_raw', 'screen_size_inch', 'screen_resolution', 'screen_type', 'weight_raw', 'weight_kg', 'battery', 'os', 'ports', 'key_selling_points', 'promotions', 'warranty', 'usage_tags', 'product_url', 'image_url', 'ram_type_raw', 'storage_raw']


In [12]:
def clean_text(value):
    if not isinstance(value, str):
        return value

    # [THÊM] Chuẩn hóa Unicode (NFC)
    value = unicodedata.normalize("NFC", value)

    # Chuẩn hóa các thẻ xuống dòng HTML
    value = re.sub(
        r"<\s*br\s*/?\s*>",
        "\n",
        value,
        flags=re.IGNORECASE,
    )

    # Loại bỏ các thẻ HTML còn lại
    value = re.sub(r"<[^>]+>", " ", value)

    # Giải mã HTML entities
    value = unescape(value)

    # [THÊM] Bỏ ký tự zero-width và đổi nbsp thành khoảng trắng thường
    value = re.sub(r"[\u200b\u200c\u200d\u2060\ufeff]", "", value)
    value = value.replace("\xa0", " ")

    # Chuẩn hóa khoảng trắng trên từng dòng
    lines = [
        re.sub(r"[ \t]+", " ", line).strip()
        for line in value.splitlines()
    ]

    # Bỏ dòng trống ở đầu/cuối và gộp các dòng trống liên tiếp
    result = "\n".join(lines)
    result = re.sub(r"\n{2,}", "\n", result)

    return result.strip()

In [13]:

TEXT_FIELDS = [
    "name",
    "sku",
    "brand",
    "price_segment",
    "cpu",
    "ram",
    "ram_type",
    "storage",
    "gpu",
    "gpu_type",
    "screen_size_raw",
    "screen_resolution",
    "screen_type",
    "weight_raw",
    "battery",
    "os",
    "ports",
    "key_selling_points",
]


In [14]:
for laptop in laptops:
    laptop.pop('promotions', None)
    laptop.pop('product_url', None)
    laptop.pop('image_url', None)
    laptop.pop('usage_tags', None)
    laptop.pop('key_selling_points', None)
    laptop.pop('storage_raw', None)
    laptop.pop('weight_raw', None)
    laptop.pop('screen_size_raw', None)
    for field in TEXT_FIELDS:
        if field in laptop:
            laptop[field] = clean_text(laptop[field])

In [15]:

laptops_before = laptops.copy()

def laptop_invalid(laptop):
    missing_price = laptop.get("current_price") is None

    ram = laptop.get("ram")
    ram_gb = laptop.get("ram_gb")

    missing_ram = (
        ram is None
        or not str(ram).strip()
        or ram_gb is None
    )

    return missing_price or missing_ram

laptops = [
    laptop for laptop in laptops
    if not laptop_invalid(laptop)
]

print(len(laptops))


365


In [16]:
with open("laptops_cleaned.json", "w", encoding="utf-8") as file:
    json.dump(
        laptops,
        file,
        ensure_ascii=False,
        indent=4
    )